# VITAL Quickstart Tutorial

This notebook provides a minimal end-to-end example of using VITAL to:

1. Load tidal resource data
2. Load rotor performance data
3. Simulate turbine performance
4. Check key constraints
5. Estimate levelized cost of energy (LCOE)

This tutorial is intended as a starting point for new users.

The example uses a representative Sitkana-style battery-charging configuration. More detailed module-specific tutorials and case studies are provided separately.

## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from vital.module_tidal import process_tidal_data
from vital.module_rotor import RotorData
from vital.module_rotor_simulation import RotorSimulation
from vital.module_vessel import VesselData
from vital.module_constraint_checker import ConstraintChecker
from vital.module_lcoe import LCOEData, LCOECalculator

import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

## 2. Load tidal resource data

This quickstart first attempts to retrieve tidal resource data from NOAA. If NOAA services are unavailable, the workflow can fall back to a local tidal data file.

For this example, both the NOAA station and local fallback dataset correspond to station ``COI0303`` near Port Mackenzie.

The local fallback file should contain at least two columns:

- ``t``: time in seconds
- ``U``: flow speed in m/s

Note that the local file is only used if NOAA retrieval fails. If NOAA succeeds, ``tidal.source`` will be ``"NOAA"`` even when ``local_file`` and ``fallback_metadata`` are provided.

In [ ]:
tidal = process_tidal_data(
    station="COI0303",
    startdate="2021-09-01",
    range_hrs=14 * 24,
    time_step_s=3600,
    city_data_file="../data/AlaskaCityLatLong.txt",
    local_file="../data/TidalData/COI0303_20210901.txt",
    fallback_metadata={
        "station_name": "COI0303",
        "nearest_city": "Port Mackenzie",
        "latitude_deg": 61.2522,
        "longitude_deg": -149.9207,
        "mooring_distance_m": 50.0,
        "cable_length_m": 1000.0,
    },
)

print("Data source:", tidal.source)
if tidal.source == "local_file":
    print("NOAA retrieval failed, so the local fallback file was used.")
else:
    print("NOAA retrieval succeeded, so the local fallback file was not used.")
print("Station name:", tidal.station_name)
print("Nearest city:", tidal.nearest_city)
print("Mooring distance (m):", tidal.mooring_distance)
print("Cable length (m):", tidal.cable_length)
print("Latitude (deg):", np.degrees(tidal.latitude))
print("Longitude (deg):", np.degrees(tidal.longitude))
print("Number of time points:", len(tidal.times))
print("First 5 flow speeds:", tidal.flow_speeds[:5])

plt.figure(figsize=(8, 4))
plt.plot(tidal.times / 3600, tidal.flow_speeds)
plt.xlabel("Time (hours)")
plt.ylabel("Flow speed (m/s)")
plt.title(f"Tidal resource at {tidal.station_name} ({tidal.source})")
plt.grid(True)
plt.tight_layout()
plt.show()

## 3. Load and visualize rotor performance data

The rotor performance file provides:

- ``TSR``: tip-speed ratio
- ``Ct``: thrust coefficient
- ``Cq``: torque coefficient

VITAL computes the power coefficient internally as:

$$
C_p = TSR \cdot C_q
$$

If no Cpmin file is supplied, VITAL assumes:

$$
C_{p,\min} = -1.0
$$

The code below loads the rotor data, prints key summary values, and visualizes the performance curves over the measured TSR range.

In [ ]:
rotor = RotorData(
    filename="../data/Sitkana_rotor_data_blade_2.txt"
)

print(f"Optimal Cp: {rotor.CpOpt:.4f}")
print(f"Optimal TSR: {rotor.TSROpt:.4f}")
print(f"TSR max: {rotor.TSRmax:.4f}")

# Plot over the measured TSR range to avoid emphasizing extrapolated behavior.
tsr_vals = np.linspace(rotor.tsr.min(), rotor.tsr.max(), 200)

plt.figure(figsize=(8, 6))

plt.subplot(4, 1, 1)
plt.plot(tsr_vals, rotor.get_cp(tsr_vals))
plt.ylabel("Cp")

plt.subplot(4, 1, 2)
plt.plot(tsr_vals, rotor.get_cq(tsr_vals))
plt.ylabel("Cq")

plt.subplot(4, 1, 3)
plt.plot(tsr_vals, rotor.get_ct(tsr_vals))
plt.ylabel("Ct")

plt.subplot(4, 1, 4)
plt.plot(tsr_vals, rotor.get_cpmin(tsr_vals))
plt.ylabel("Cpmin")
plt.xlabel("TSR")

plt.tight_layout()
plt.show()

## 4. Define turbine configuration

This example uses a simple baseline turbine configuration for demonstration. The configuration below connects the tidal resource and rotor performance data to the rotor simulation model.

This quickstart uses the built-in ``simple`` power-conversion model, which requires the generator constants ``Kt`` and ``Rw``.

In [ ]:
config = {
    'Radius': 0.5,
    'Prated': 2500.0,
    'Trated': np.inf,
    'dHub': 2.0,
    'number_of_turbines': 2,
    'dMoor': tidal.mooring_distance,
    'Uinf': tidal.flow_speeds,
    't': tidal.times,
    'CpFunc': rotor.get_cp,
    'CqFunc': rotor.get_cq,
    'CtFunc': rotor.get_ct,
    'CpOpt': rotor.CpOpt,
    'TSROpt': rotor.TSROpt,
    'TSRmax': rotor.TSRmax,
    'Ng': 20,
    'Kt': 1.5,
    'Rw': 0.5,
    'J_d': 1,
    'B_d': 0.01,
    'J_r': 10,
    'power_model': 'simple',
}

print("Turbine configuration defined.")
print(f"Radius (m): {config['Radius']}")
print(f"Rated power (W): {config['Prated']}")
print(f"Hub depth (m): {config['dHub']}")
print(f"Number of turbines: {config['number_of_turbines']}")

## 5. Run rotor simulation and visualize outputs

The rotor simulation uses the tidal resource, rotor performance curves, and turbine configuration to estimate turbine speed, torque, thrust, and power over time. The plots below summarize the main simulation outputs.

For the ``simple`` power model, ``Pelec`` is the electrical power output from the built-in generator model.

In [ ]:
rotor_sim = RotorSimulation(config)
rotor_sim.simulate()
result = rotor_sim.get_results()

print("Rotor simulation complete.")
print("Available result keys:", list(result.keys()))

fig, ax = plt.subplots(3, 2, figsize=(12, 10), sharex=True)

time_days = result["t"] / (24 * 3600)

ax[0, 0].plot(time_days, result["Uinf_adjusted"])
ax[0, 0].set_ylabel("Flow (m/s)")
ax[0, 0].set_title("Hub-depth flow")
ax[0, 0].grid(True)

ax[0, 1].plot(time_days, result["w"])
ax[0, 1].set_ylabel("Generator speed (rad/s)")
ax[0, 1].set_title("Generator-side speed")
ax[0, 1].grid(True)

ax[1, 0].plot(time_days, result["Th"])
ax[1, 0].set_ylabel("Torque (N m)")
ax[1, 0].set_title("Hydrodynamic torque")
ax[1, 0].grid(True)

ax[1, 1].plot(time_days, result["Tg"])
ax[1, 1].set_ylabel("Torque (N m)")
ax[1, 1].set_title("Generator torque")
ax[1, 1].grid(True)

ax[2, 0].plot(time_days, result["Pelec"] / 1000, label="Electrical power")
ax[2, 0].axhline(config["Prated"] / 1000, color="r", linestyle="--", label="Rated power")
ax[2, 0].set_ylabel("Power (kW)")
ax[2, 0].set_title("Electrical power")
ax[2, 0].set_xlabel("Time (days)")
ax[2, 0].grid(True)
ax[2, 0].legend()

ax[2, 1].plot(time_days, result["TSR"])
ax[2, 1].set_ylabel("TSR (-)")
ax[2, 1].set_title("Tip-speed ratio")
ax[2, 1].set_xlabel("Time (days)")
ax[2, 1].grid(True)

plt.tight_layout()
plt.show()

## 6. Define representative vessel properties

The vessel properties below represent a simplified, user-defined example for constraint checking and LCOE estimation. Users may replace these values with site- or vessel-specific data.

In [ ]:
user_vessel_properties = {
    'Xm': 5.77,
    'Zm': 1.65,
    'Kphi': 1.95e6,
    'theta': np.radians(45.0),
    'phi': np.radians(20.0),
    'area': 9.5,
    'Cd': 1.0,
}

vessel = VesselData(
    user_defined=True,
    vessel_properties=user_vessel_properties,
    simResult=result
)

print(f"Maximum vessel drag force: {np.max(vessel.Fdrag):.2f} N")
print(f"Mean vessel drag force: {np.mean(vessel.Fdrag):.2f} N")

plt.figure(figsize=(8, 4))
plt.plot(result["t"] / 3600, vessel.Fdrag)
plt.xlabel("Time (hours)")
plt.ylabel("Vessel drag force (N)")
plt.title("Representative Vessel Drag Force")
plt.grid(True)
plt.tight_layout()
plt.show()

## 7. Check constraints

The constraint checker evaluates whether the turbine and vessel configuration satisfies key operating limits, including power, depth, cavitation, and pitch stability constraints.

In [ ]:
constraint_checker = ConstraintChecker(rotor, config, vessel, result)

constraints = {
    "Power Constraint": constraint_checker.check_power_constraint(),
    "Depth Constraint": constraint_checker.check_depth_constraint(),
    "Cavitation Constraint": constraint_checker.check_cavitation_constraint(),
    "Pitch Constraint": constraint_checker.check_pitch_constraint(),
}

print("Constraint Results:")
for name, satisfied in constraints.items():
    status = "Satisfied" if satisfied else "Not Satisfied"
    print(f"  {name}: {status}")

print()
print("Constraint margins:")
print(f"  Minimum power margin, Prated - Pelec: {np.min(constraint_checker.power_constraint()):.3f} W")
print(f"  Depth margin, dHub - Radius: {constraint_checker.depth_constraint():.3f} m")
print(f"  Minimum cavitation margin: {np.min(constraint_checker.cavitation_constraint()):.3f} Pa")
print(f"  Minimum pitch margin: {np.min(constraint_checker.pitch_constraint()):.3f} N m")

if not all(constraints.values()):
    print()
    print("Warning: One or more constraints are not satisfied. LCOE can still be calculated, but the design should be treated as infeasible.")

## 8. Estimate LCOE

The LCOE calculation combines the tidal resource, turbine configuration, vessel properties, and simulation results to estimate annual energy production and levelized cost of energy.

This quickstart uses:

```python
customer = "customer_B"
application = "battery_charging"
BatteryCapacity_kWh = 10.0
```

Because this is a battery-charging application, ``BatteryCapacity_kWh`` must be provided and must be greater than zero.

In [ ]:
data = LCOEData(
    tidalData=tidal,
    turbineConfig=config,
    vesselData=vessel,
    simResult=result,
    lifetime=10,
    discount_rate=0.1,
    turbulence_intensity=0.0,
    customer="customer_B",
    application="battery_charging",
    BatteryCapacity_kWh=10.0,
)

calculator = LCOECalculator(data)

total_capex = calculator.calculate_total_capex()
total_opex = calculator.calculate_total_opex(total_capex)
annual_energy = calculator.calculate_annual_energy()
lcoe = calculator.calculate_lcoe()

print(f"LCOE: ${lcoe:.4f}/kWh")
print(f"Annual energy: {annual_energy:.2f} kWh")
print(f"Total CAPEX: ${total_capex:,.2f}")
print(f"Total OPEX: ${total_opex:,.2f} per year")

## 9. Show cost breakdown

The cost breakdown shows the main CAPEX and OPEX components used in the LCOE calculation, along with the annual energy estimate.

In [ ]:
calculator.list_capex()
calculator.list_opex(total_capex)
calculator.list_annual_energy()

## 10. Summary

This tutorial demonstrated a basic VITAL workflow, including:

- loading and using resource data
- loading and visualizing rotor performance curves
- running a turbine simulation using a representative configuration
- checking vessel and operating constraints
- estimating LCOE and reporting cost breakdowns

This workflow is designed to be adaptable to different resource sites, rotor performance curves, vessel configurations, and cost-model assumptions.